[◀ Notebook 01: Lexical Structure](01_lexical_structure.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 03: Expressions ▶](03_expressions.ipynb)

# Notebook 02: The Python Data Model (Fundamentals)

In Python, *everything* is an object. Python's data model defines the rules governing how objects interact, how memory is allocated, and how references function.

Official Reference: [Python Data Model](https://docs.python.org/3/reference/datamodel.html)

## 1. Identity, Type, and Value

Every object in Python has three fundamental components:

1.  **Identity:** An integer representing the object's address in memory. In CPython, this is the virtual memory address where the object resides. It is retrieved using `id(obj)` and is guaranteed to be unique and constant for the object's lifetime.
2.  **Type:** Defines the operations the object supports and the possible values it can hold. It is retrieved using `type(obj)` and, once set, is generally immutable.
3.  **Value:** The actual data stored in the object.

### Identity vs. Equality
*   The `==` operator compares the **values** of two objects.
*   The `is` operator compares the **identities** of two objects. It evaluates to `True` if and only if both variables point to the exact same memory location (`id(a) == id(b)`).

In [ ]:
# Comparison of identity vs equality
x = [1, 2, 3]
y = [1, 2, 3]
z = x

print("x == y:", x == y)   # True: Values are identical
print("x is y:", x is y)   # False: Separate objects in memory
print("x is z:", x is z)   # True: Both variables bind to the same object
print("id(x):", id(x))
print("id(y):", id(y))
print("id(z):", id(z))

## 2. Mutability vs. Immutability

An object's type determines whether its value can change.

*   **Immutable Objects:** The value cannot change once created. Examples include numbers (`int`, `float`), strings (`str`), tuples (`tuple`), and frozen sets (`frozenset`). Any operation that seems to alter them actually creates a new object with a new identity.
*   **Mutable Objects:** The value can change in-place without changing its identity. Examples include `list`, `dict`, `set`, and user-defined classes.

### Mutability Side-Effects
When multiple names refer to the same mutable object, mutating the object via one reference affects all references. This is a common source of bugs.

### Shallow vs. Deep Copying
To avoid side-effects when working with mutable objects, Python provides copying tools via the `copy` module:
-   **Shallow Copy (`copy.copy(obj)`):** Creates a new container object, but populates it with references to the nested objects found inside the original. If you modify a nested mutable structure (like modifying a dictionary inside a list), that change propagates to both lists.
-   **Deep Copy (`copy.deepcopy(obj)`):** Recursively creates a new container and duplicates all nested objects. Modifying nested mutable structures in a deep-copied object has no effect on the original.
-   **CPython Memory Reference Propagation:** A shallow copy constructs a new outer container but copies the exact same memory pointers of the elements it holds. A deep copy recursively duplicates every object it encounters, tracking copies in an internal `memo` dictionary to prevent infinite recursion on self-referential structures.

### CPython Operator Identity: `+` vs `+=`
For sequence types like lists, the choice of operator changes memory behavior:
-   **In-Place Operator (`+=`):** Calls the `__iadd__` magic method if implemented. For lists, `L += [item]` mutates the list in-place (equivalent to `L.extend([item])`). The list's virtual memory address (returned by `id(L)`) remains **identical**.
-   **Concatenation Operator (`+`):** Calls the `__add__` magic method. For lists, `L = L + [item]` creates a **new list** containing elements from both operands, then rebinds the name `L` to this new object. The object's ID **changes**.

In [ ]:
# 1. Immutable reassignment
text = "Hello"
print("Initial text ID:", id(text))
text = text + " World" # Creates a brand new string object!
print("After concat ID:", id(text))

# 2. Mutable mutation side-effects
shared_list = [10, 20]
alias = shared_list
alias.append(30) # Mutates the object shared_list points to!
print("\nOriginal list after alias mutation:", shared_list)

import copy

# 3. Shallow Copy (nested objects are shared by reference)
original_nested = [[1, 2], [3, 4]]
shallow_copied = copy.copy(original_nested)
shallow_copied[0].append(99)
print("\nShallow Copy behavior (mutating nested element):")
print("Original nested:", original_nested)
print("Shallow copied:", shallow_copied)

# 4. Deep Copy (nested objects are recursively copied)
original_nested_2 = [[1, 2], [3, 4]]
deep_copied = copy.deepcopy(original_nested_2)
deep_copied[0].append(99)
print("\nDeep Copy behavior (mutating nested element):")
print("Original nested 2:", original_nested_2)
print("Deep copied:", deep_copied)

# 5. CPython Operator Identity: + vs += on lists
list_add = [1, 2]
id_add_before = id(list_add)
list_add = list_add + [3, 4]  # Creates a new list object
print("\nList + Operator:")
print("ID remained same?", id(list_add) == id_add_before)

list_iadd = [1, 2]
id_iadd_before = id(list_iadd)
list_iadd += [3, 4]  # Mutates the list in-place (calling __iadd__)
print("List += Operator:")
print("ID remained same?", id(list_iadd) == id_iadd_before)


## 3. CPython Memory Interning (Under the Hood)

To optimize performance and reduce memory fragmentation, the CPython implementation of Python uses **interning** (object caching) for commonly used immutable values.

### Small Integer Interning
CPython pre-allocates and caches all integers between **`-5` and `256`** inclusive. When you bind a variable to an integer in this range, it points to the pre-allocated cached object. Integers outside this range are generally allocated dynamically on demand.

### String Interning
Short strings that resemble Python identifiers (alphanumeric and underscores) are automatically interned. This speeds up dictionary key lookups because checking identifier equality simplifies to a fast pointer comparison (`is`) rather than character-by-character validation.

In [ ]:
# Testing integer interning
a1 = 256
b1 = 256
print("256 is interned:", a1 is b1) # True

# We construct the integer dynamically to avoid compile-time constant folding optimization
a2 = 250 + 7
b2 = 260 - 3
print("257 is interned:", a2 is b2) # False (typically)

# Testing string interning
s1 = "python_identifier"
s2 = "python_identifier"
print("Identifier-like string is interned:", s1 is s2) # True

---

## Coding Challenges

Complete the exercises below to verify your understanding of Python's objects and memory models.

In [ ]:
# ==========================================
# Challenge 1: Find Interning Bounds Programmatically
# ==========================================
# Instructions: Write a function `find_interning_bounds()` that returns a tuple `(lower, upper)`
# indicating the exact range [lower, upper] of integers that CPython interns.
# 
# To prevent Python's compiler from optimizing constants (e.g., if you write `a = 500; b = 500` in the
# same block, the compiler folds them into one object), you must construct the integer objects dynamically
# (e.g., using mathematical expressions, int(string), or a function returning the number).

def find_interning_bounds() -> tuple:
    # TODO: Implement. Scan integers dynamically in a loop around the known limits.
    pass

In [ ]:
# Test assertions for Challenge 1
try:
    lower, upper = find_interning_bounds()
    assert lower == -5, f"Expected lower bound of -5, got {lower}"
    assert upper == 256, f"Expected upper bound of 256, got {upper}"
    print("🎉 Challenge 1 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 1 Failed: {e}")
except TypeError:
    print("❌ Function must return a tuple (lower, upper)")

In [ ]:
# ==========================================
# Challenge 2: Mutability Side-Effects Guard
# ==========================================
# Instructions: Write a function `safe_add_employee(employee_db, new_employee)`
# that appends `new_employee` (a dict) to `employee_db` (a list of dicts) and returns
# the updated database list.
# 
# Crucially, to isolate data structures and avoid bugs:
# 1. The original list passed to `employee_db` must NOT be modified.
# 2. None of the dict objects inside the original list, nor the `new_employee` dict, 
#    should be shared by reference. They must be copied so that modifying the returned list's 
#    contents later does not alter the objects in the original list.

import copy

def safe_add_employee(employee_db: list, new_employee: dict) -> list:
    # TODO: Implement the safe, side-effect-free database update.
    pass

In [ ]:
# Test assertions for Challenge 2
try:
    db = [
        {"id": 1, "name": "Alice", "role": "Developer"},
        {"id": 2, "name": "Bob", "role": "Designer"}
    ]
    new_emp = {"id": 3, "name": "Charlie", "role": "Manager"}
    
    updated_db = safe_add_employee(db, new_emp)
    
    # Verify size and identity of the parent list
    assert len(db) == 2, "Original list was mutated in-place!"
    assert len(updated_db) == 3, "New employee was not added!"
    
    # Verify dictionary independence (no sharing references)
    assert updated_db[0] is not db[0], "Dictionary objects in the list must be separate copies!"
    assert updated_db[2] is not new_emp, "New employee dictionary should be copied!"
    
    # Verify contents remain equal
    assert updated_db[0] == db[0], "Values of the original records changed!"
    
    # Mutate updated_db records to check separation
    updated_db[0]["role"] = "Lead Developer"
    assert db[0]["role"] == "Developer", "Modifying the copy changed the original database!"
    
    print("🎉 Challenge 2 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 2 Failed: {e}")

In [ ]:
# ==========================================
# Challenge 3: In-Place vs Out-of-Place Identity Tracking
# ==========================================
# Instructions: Write a function `compare_operation_identities()` that performs
# two different concatenation operations on a list:
# 1. In-place concatenation using `+=`
# 2. Out-of-place concatenation using `+`
#
# Specifically:
# - Create a list `L = [1, 2]`.
# - Record its initial ID.
# - Perform `L += [3]`. Record if ID remains identical (True/False).
# - Perform `L = L + [4]`. Record if ID remains identical (True/False) relative to the ID right before this statement.
# Return a tuple of two booleans: (id_same_after_iadd, id_same_after_add).

def compare_operation_identities() -> tuple:
    # TODO: Implement
    pass

In [ ]:
# Test assertions for Challenge 3
try:
    iadd_same, add_same = compare_operation_identities()
    assert iadd_same == True, "List += should modify in-place (same ID)"
    assert add_same == False, "List = List + List should create a new object (new ID)"
    print("🎉 Challenge 3 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 3 Failed: {e}")

[◀ Notebook 01: Lexical Structure](01_lexical_structure.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 03: Expressions ▶](03_expressions.ipynb)